<a href="https://colab.research.google.com/github/ShuriKimura0402/Astronomy_scripts/blob/main/DevRel%E6%97%A5%E8%A8%9810%20step1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 依存ライブラリのインストールと NeMo のセットアップ

In [1]:
!pip install nemo
!pip install nemo_toolkit[asr]
!git clone -b v2.7.2 https://github.com/NVIDIA/NeMo.git
!pip install "datasets==2.20.0" "huggingface_hub==0.23.5" "fsspec==2024.5.0" soundfile librosa
!pip install transformers -U
!pip install pyctcdecode
# Numbaとの互換性を保つため、NumPy 2.2以下のバージョンを強制インストールします
!pip install "numpy>=2.0.0,<2.3.0" -U

  Using cached nemo_toolkit-3.0.0-py3-none-any.whl.metadata (38 kB)
  Using cached aistore-2.0.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached onnx-1.23.2-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.7 kB)
  Using cached braceexpand-0.1.7-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached kaldialign-0.12.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (6.1 kB)
  Using cached lhotse-1.33.0-py3-none-any.whl.metadata (21 kB)
  Using cached sacrebleu-2.6.0-py3-none-any.whl.metadata (39 kB)
  Using cached whisper_normalizer-0.1.15-py3-none-any.whl.metadata (8.3 kB)
  Using cached hydra_core-1.3.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached lightning-2.4.0-py3-none-any.whl.metadata (38 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 58.6 MB/s eta 0:00:00
  

Cloning into 'NeMo'...
remote: Enumerating objects: 312693, done.
remote: Counting objects: 100% (966/966), done.
remote: Compressing objects: 100% (427/427), done.
remote: Total 312693 (delta 749), reused 578 (delta 539), pack-reused 311727 (from 2)
Receiving objects: 100% (312693/312693), 520.12 MiB | 19.87 MiB/s, done.
Resolving deltas: 100% (228780/228780), done.
Note: switching to '3d73c48aca1ae3be44657267b81f25dc3201161a'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

INFO: pip is looking at multiple versions of multi

  Using cached numpy-2.2.6-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nemo-toolkit 3.0.0 requires fsspec>=2024.12.0, but you have fsspec 2024.5.0 which is incompatible.
pyctcdecode 0.5.0 requires numpy<2.0.0,>=1.15.0, but you have numpy 2.2.6 which is incompatible.


# 2. Google Drive のマウント

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 3. データセットのダウンロードと前処理

In [3]:
import json, os, unicodedata
from collections import Counter
import soundfile as sf
from datasets import load_dataset
from tqdm import tqdm

OUT_DIR    = "/content/reazonspeech_small_wav"
MANIFEST   = "/content/manifest_all.json"
LABELS_FILE= "/content/labels.json"
os.makedirs(OUT_DIR, exist_ok=True)

# ★処理時間を短縮するため、まずは合計約3時間分に制限して動作確認します（制限なしの場合は None）
MAX_HOURS = 3.0
MIN_DUR, MAX_DUR = 0.5, 20.0

ds = load_dataset("reazon-research/reazonspeech", "small", trust_remote_code=True)

counter, entries, total = Counter(), [], 0.0
n_broken, n_skipped = 0, 0
broken_ids = []

for i, sample in enumerate(tqdm(ds["train"])):
    wav_path = os.path.abspath(os.path.join(OUT_DIR, f"{i:06d}.wav"))

    try:
        text = unicodedata.normalize("NFKC", sample["transcription"] or "").strip()
        if not text:
            n_skipped += 1
            continue

        if os.path.exists(wav_path):
            info = sf.info(wav_path)
            dur = info.frames / info.samplerate
        else:
            audio = sample["audio"]["array"]
            sr    = sample["audio"]["sampling_rate"]
            dur   = len(audio) / sr
            if not (MIN_DUR <= dur <= MAX_DUR):
                n_skipped += 1
                continue
            sf.write(wav_path, audio, sr)

    except Exception as e:
        n_broken += 1
        broken_ids.append(i)
        if n_broken <= 5:
            print(f"\n[skip] index={i}: {type(e).__name__}: {e}")
        if os.path.exists(wav_path):
            try: os.remove(wav_path)
            except OSError: pass
        continue

    if not (MIN_DUR <= dur <= MAX_DUR):
        n_skipped += 1
        continue

    entries.append({"audio_filepath": wav_path, "duration": round(dur, 3), "text": text})
    counter.update(text)
    total += dur

    if MAX_HOURS and total / 3600 >= MAX_HOURS:
        print(f"設定された制限時間 {MAX_HOURS} 時間に達したため、処理を終了します。")
        break

print(f"\n壊れて飛ばした: {n_broken} 件  {broken_ids[:20]}")
print(f"条件で除外した: {n_skipped} 件（無音/長すぎ/テキスト空）")

MIN_CHAR_FREQ = 5
labels = sorted([c for c, n in counter.items() if n >= MIN_CHAR_FREQ])
label_set = set(labels)

clean = [e for e in entries if set(e["text"]) <= label_set]

with open(MANIFEST, "w", encoding="utf-8") as f:
    for e in clean:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")
with open(LABELS_FILE, "w", encoding="utf-8") as f:
    json.dump(labels, f, ensure_ascii=False)

print(f"発話数: {len(entries)} → OOV除去後: {len(clean)}")
print(f"総時間: {total/3600:.1f} 時間")
print(f"語彙サイズ: {len(labels)}")

Generating train split: 0 examples [00:00, ? examples/s]

  3%|▎         | 1947/62047 [00:28<14:29, 69.09it/s] 

設定された制限時間 3.0 時間に達したため、処理を終了します。

壊れて飛ばした: 0 件  []
条件で除外した: 17 件（無音/長すぎ/テキスト空）
発話数: 1931 → OOV除去後: 1025
総時間: 3.0 時間
語彙サイズ: 802


# 4. 訓練データと検証データの分割

In [4]:
import json, random
random.seed(42)

lines = open("/content/manifest_all.json", encoding="utf-8").read().splitlines()
random.shuffle(lines)
n_val = max(200, int(len(lines) * 0.02))
val, train = lines[:n_val], lines[n_val:]

for name, rows in [("train", train), ("val", val)]:
    with open(f"/content/manifest_{name}.json", "w", encoding="utf-8") as f:
        f.write("\n".join(rows) + "\n")
    print(name, len(rows))

train 825
val 200


In [1]:
!pip install nemo
!pip install nemo_toolkit[asr]
!git clone -b v2.7.2 https://github.com/NVIDIA/NeMo.git
!pip install "datasets==2.20.0" "huggingface_hub==0.23.5" "fsspec==2024.5.0" soundfile librosa
!pip install transformers -U
# ★推論でN-best(beam)を使うなら追加。kenlm は入れない（ビルドが必要で失敗する）
!pip install pyctcdecode

from google.colab import drive
drive.mount('/content/drive')

  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached pyarrow-25.0.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached fsspec-2025.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
Using cached datasets-5.1.0-py3-none-any.whl (604 kB)
Using cached fsspec-2025.12.0-py3-none-any.whl (201 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)
Using cached pyarrow-25.0.1-cp313-cp313-manylinux_2_28_x86_64.whl (50.1 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: packaging
    Found existing installation: packaging 26.3
    Uninstalling packaging-26.3:
      Successfully uninstalled packaging-26.3
  Attempting uninstall: fsspec

fatal: destination path 'NeMo' already exists and is not an empty directory.
  Using cached datasets-2.20.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-0.23.5-py3-none-any.whl.metadata (12 kB)
  Using cached fsspec-2024.5.0-py3-none-any.whl.metadata (11 kB)
Using cached datasets-2.20.0-py3-none-any.whl (547 kB)
Using cached huggingface_hub-0.23.5-py3-none-any.whl (402 kB)
Using cached fsspec-2024.5.0-py3-none-any.whl (316 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: datasets
    Found existing installation: datasets 5.1.0
    Uninstalling datasets-5.1.0:
      Successfully uninstalled datasets-5.1.0
ERROR: pip's dependency resolver d

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
PREP = r'''
import json, os, unicodedata
from collections import Counter
import soundfile as sf
from datasets import load_dataset
from tqdm import tqdm

OUT_DIR    = "/content/reazonspeech_small_wav"
MANIFEST   = "/content/manifest_all.json"
LABELS_FILE= "/content/labels.json"
os.makedirs(OUT_DIR, exist_ok=True)

# 絞りたいときは時間数で上限（ディスク/時間が厳しいとき用。Noneで全部）
MAX_HOURS = None
MIN_DUR, MAX_DUR = 0.5, 20.0      # 極端に短い/長い発話は学習が不安定・OOMの元

ds = load_dataset("reazon-research/reazonspeech", "small", trust_remote_code=True)

counter, entries, total = Counter(), [], 0.0
n_broken, n_skipped = 0, 0
broken_ids = []

for i, sample in enumerate(tqdm(ds["train"])):
    wav_path = os.path.abspath(os.path.join(OUT_DIR, f"{i:06d}.wav"))

    # ★壊れた音声が混ざっていても止まらないようにする
    #   （ReazonSpeechにはデコードできないFLACが少数ある。1本で全部やり直しは避けたい）
    try:
        text = unicodedata.normalize("NFKC", sample["transcription"] or "").strip()
        if not text:
            n_skipped += 1
            continue

        if os.path.exists(wav_path):
            # 既に書き出し済み → デコードし直さない（再実行がぐっと速くなる）
            info = sf.info(wav_path)
            dur = info.frames / info.samplerate
        else:
            audio = sample["audio"]["array"]        # ここでFLACがデコードされる
            sr    = sample["audio"]["sampling_rate"]
            dur   = len(audio) / sr
            if not (MIN_DUR <= dur <= MAX_DUR):
                n_skipped += 1
                continue
            sf.write(wav_path, audio, sr)

    except Exception as e:
        # 壊れた音声・読めないサンプルは捨てて次へ
        n_broken += 1
        broken_ids.append(i)
        if n_broken <= 5:
            print(f"\n[skip] index={i}: {type(e).__name__}: {e}")
        if os.path.exists(wav_path):               # 書きかけのファイルは消す
            try: os.remove(wav_path)
            except OSError: pass
        continue

    if not (MIN_DUR <= dur <= MAX_DUR):
        n_skipped += 1
        continue
    if MAX_HOURS and total / 3600 >= MAX_HOURS:
        break

    entries.append({"audio_filepath": wav_path, "duration": round(dur, 3), "text": text})
    counter.update(text)
    total += dur

print(f"\n壊れて飛ばした: {n_broken} 件  {broken_ids[:20]}")
print(f"条件で除外した: {n_skipped} 件（無音/長すぎ/テキスト空）")

# 語彙: tinyは「2回以上」で足りたが、smallは漢字が増えるので閾値を上げる
MIN_CHAR_FREQ = 5
labels = sorted([c for c, n in counter.items() if n >= MIN_CHAR_FREQ])
label_set = set(labels)

# ★OOV(語彙外の文字)を含む発話は落とす。残すとCTCの学習時に落ちる/ノイズになる
clean = [e for e in entries if set(e["text"]) <= label_set]

with open(MANIFEST, "w", encoding="utf-8") as f:
    for e in clean:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")   # ensure_ascii=False で日本語そのまま
with open(LABELS_FILE, "w", encoding="utf-8") as f:
    json.dump(labels, f, ensure_ascii=False)

print(f"発話数: {len(entries)} → OOV除去後: {len(clean)}")
print(f"総時間: {total/3600:.1f} 時間")
print(f"語彙サイズ: {len(labels)}  (漢字が十分入っているか目視で確認)")
print("サンプル:", labels[:40])
'''

In [3]:
SPLIT = r'''
import json, random
random.seed(42)

lines = open("/content/manifest_all.json", encoding="utf-8").read().splitlines()
random.shuffle(lines)
n_val = max(200, int(len(lines) * 0.02))        # 2% を検証用に
val, train = lines[:n_val], lines[n_val:]

for name, rows in [("train", train), ("val", val)]:
    with open(f"/content/manifest_{name}.json", "w", encoding="utf-8") as f:
        f.write("\n".join(rows) + "\n")
    print(name, len(rows))
'''

In [4]:
!rm -rf /root/.cache/huggingface
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   45G  192G  19% /


In [5]:

TRAIN = r'''
import json, os
with open("/content/labels.json", encoding="utf-8") as f:
    labels = json.load(f)
os.environ["LABELS"] = "[" + ",".join(f"'{c}'" for c in labels) + "]"
print("語彙サイズ:", len(labels))

!mkdir -p /content/drive/MyDrive/nest_ja_small

!python /content/NeMo/examples/asr/asr_ctc/speech_to_text_ctc.py \
    --config-path=/content/NeMo/examples/asr/conf/conformer/ \
    --config-name=conformer_ctc_char \
    ++init_from_pretrained_model.ssl.name="sbintuitions/nest-ja-0.1b" \
    ++init_from_pretrained_model.ssl.include=["encoder"] \
    model.encoder.n_layers=17 \
    model.encoder.d_model=512 \
    model.encoder.subsampling="dw_striding" \
    model.encoder.subsampling_factor=8 \
    model.encoder.subsampling_conv_channels=256 \
    model.encoder.conv_kernel_size=9 \
    model.encoder.conv_norm_type="layer_norm" \
    model.encoder.xscaling="true" \
    model.train_ds.manifest_filepath=/content/manifest_train.json \
    model.validation_ds.manifest_filepath=/content/manifest_val.json \
    "model.labels=$LABELS" \
    model.train_ds.batch_size=32 \
    model.train_ds.num_workers=8 \
    model.train_ds.max_duration=20.0 \
    model.train_ds.min_duration=0.5 \
    model.train_ds.shuffle=true \
    model.validation_ds.batch_size=32 \
    model.validation_ds.num_workers=4 \
    trainer.devices=-1 \
    trainer.accelerator="gpu" \
    trainer.strategy="auto" \
    trainer.precision="bf16-mixed" \
    trainer.max_epochs=50 \
    trainer.val_check_interval=0.5 \
    model.optim.name="adamw" \
    model.optim.lr=0.0003 \
    model.optim.betas=[0.9,0.999] \
    model.optim.weight_decay=0.0001 \
    model.optim.sched.warmup_steps=5000 \
    exp_manager.exp_dir=/content/drive/MyDrive/nest_ja_small \
    exp_manager.resume_if_exists=true \
    exp_manager.resume_ignore_no_checkpoint=true \
    exp_manager.create_checkpoint_callback=true \
    exp_manager.checkpoint_callback_params.save_top_k=3 \
    exp_manager.checkpoint_callback_params.monitor="val_wer" \
    exp_manager.checkpoint_callback_params.mode="min" \
    exp_manager.create_wandb_logger=true \
    exp_manager.wandb_logger_kwargs.name="nest-ja-small" \
    exp_manager.wandb_logger_kwargs.project="nest-ja"
'''

In [6]:
import json, os
with open("/content/labels.json", encoding="utf-8") as f:
    labels = json.load(f)
os.environ["LABELS"] = "[" + ",".join(f"'{c}'" for c in labels) + "]"
print("語彙サイズ:", len(labels))

!mkdir -p /content/drive/MyDrive/nest_ja_small

!python /content/NeMo/examples/asr/asr_ctc/speech_to_text_ctc.py \
    --config-path=/content/NeMo/examples/asr/conf/conformer/ \
    --config-name=conformer_ctc_char \
    ++init_from_pretrained_model.ssl.name="sbintuitions/nest-ja-0.1b" \
    ++init_from_pretrained_model.ssl.include=["encoder"] \
    model.encoder.n_layers=17 \
    model.encoder.d_model=512 \
    model.encoder.subsampling="dw_striding" \
    model.encoder.subsampling_factor=8 \
    model.encoder.subsampling_conv_channels=256 \
    model.encoder.conv_kernel_size=9 \
    model.encoder.conv_norm_type="layer_norm" \
    model.encoder.xscaling="true" \
    model.train_ds.manifest_filepath=/content/manifest_train.json \
    model.validation_ds.manifest_filepath=/content/manifest_val.json \
    "model.labels=$LABELS" \
    model.train_ds.batch_size=32 \
    model.train_ds.num_workers=8 \
    model.train_ds.max_duration=20.0 \
    model.train_ds.min_duration=0.5 \
    model.train_ds.shuffle=true \
    model.validation_ds.batch_size=32 \
    model.validation_ds.num_workers=4 \
    trainer.devices=-1 \
    trainer.accelerator="gpu" \
    trainer.strategy="auto" \
    trainer.precision="bf16-mixed" \
    trainer.max_epochs=50 \
    trainer.val_check_interval=0.5 \
    model.optim.name="adamw" \
    model.optim.lr=0.0003 \
    model.optim.betas=[0.9,0.999] \
    model.optim.weight_decay=0.0001 \
    model.optim.sched.warmup_steps=5000 \
    exp_manager.exp_dir=/content/drive/MyDrive/nest_ja_small \
    exp_manager.resume_if_exists=true \
    exp_manager.resume_ignore_no_checkpoint=true \
    exp_manager.create_checkpoint_callback=true \
    exp_manager.checkpoint_callback_params.save_top_k=3 \
    exp_manager.checkpoint_callback_params.monitor="val_wer" \
    exp_manager.checkpoint_callback_params.mode="min" \
    exp_manager.create_wandb_logger=true \
    exp_manager.wandb_logger_kwargs.name="nest-ja-small" \
    exp_manager.wandb_logger_kwargs.project="nest-ja"

語彙サイズ: 802
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo I 2026-10-09 03:15:00 speech_to_text_ctc:83] Hydra config: name: Conformer-CTC-Char
    model:
      sample_rate: 16000
      labels:
      - '!'
      - '%'
      - .
      - '0'
      - '1'
      - '2'
      - '3'
      - '4'
      - '5'
      - '6'
      - '7'
      - '8'
      - '9'
      - '?'
      - A
      - C
      - D
      - G
      - J
      - 'N'
      - O
      - R
      - S
      - T
      - '['
      - ']'
      - m
      - o
      - 、
      - 。
      - 々
      - 〈
      - 〉
      - 《
      - 》
      - 「
      - 」
      - ぁ
      - あ
      - い
      - う
      - え
      - お
      - か
      - が
      - き
      - ぎ
      - く
      - ぐ
      - け
      - げ
      - こ
      - ご
      - さ
    

In [1]:
# 1. NumPyのダウングレード (Numba 互換性の確保)
!pip install "numpy>=2.0.0,<2.3.0" -U

# 2. 前処理 (PREP) を直接実行します
import json, os, unicodedata
from collections import Counter
import soundfile as sf
from datasets import load_dataset
from tqdm import tqdm

OUT_DIR    = "/content/reazonspeech_small_wav"
MANIFEST   = "/content/manifest_all.json"
LABELS_FILE= "/content/labels.json"
os.makedirs(OUT_DIR, exist_ok=True)

# ディスクとトレーニング時間の確保のため、3時間に制限します
MAX_HOURS = 3.0
MIN_DUR, MAX_DUR = 0.5, 20.0

print("ReazonSpeech データセットの読み込みを開始します...")
ds = load_dataset("reazon-research/reazonspeech", "small", trust_remote_code=True)

counter, entries, total = Counter(), [], 0.0
n_broken, n_skipped = 0, 0
broken_ids = []

for i, sample in enumerate(tqdm(ds["train"])):
    wav_path = os.path.abspath(os.path.join(OUT_DIR, f"{i:06d}.wav"))

    try:
        text = unicodedata.normalize("NFKC", sample["transcription"] or "").strip()
        if not text:
            n_skipped += 1
            continue

        if os.path.exists(wav_path):
            info = sf.info(wav_path)
            dur = info.frames / info.samplerate
        else:
            audio = sample["audio"]["array"]
            sr    = sample["audio"]["sampling_rate"]
            dur   = len(audio) / sr
            if not (MIN_DUR <= dur <= MAX_DUR):
                n_skipped += 1
                continue
            sf.write(wav_path, audio, sr)

    except Exception as e:
        n_broken += 1
        broken_ids.append(i)
        if n_broken <= 5:
            print(f"\n[skip] index={i}: {type(e).__name__}: {e}")
        if os.path.exists(wav_path):
            try: os.remove(wav_path)
            except OSError: pass
        continue

    if not (MIN_DUR <= dur <= MAX_DUR):
        n_skipped += 1
        continue

    entries.append({"audio_filepath": wav_path, "duration": round(dur, 3), "text": text})
    counter.update(text)
    total += dur

    if MAX_HOURS and total / 3600 >= MAX_HOURS:
        print(f"設定された制限時間 {MAX_HOURS} 時間に達したため、抽出を終了します。")
        break

print(f"\n壊れて飛ばした: {n_broken} 件  {broken_ids[:20]}")
print(f"条件で除外した: {n_skipped} 件（無音/長すぎ/テキスト空）")

MIN_CHAR_FREQ = 5
labels = sorted([c for c, n in counter.items() if n >= MIN_CHAR_FREQ])
label_set = set(labels)

clean = [e for e in entries if set(e["text"]) <= label_set]

with open(MANIFEST, "w", encoding="utf-8") as f:
    for e in clean:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")
with open(LABELS_FILE, "w", encoding="utf-8") as f:
    json.dump(labels, f, ensure_ascii=False)

print(f"発話数: {len(entries)} → OOV除去後: {len(clean)}")
print(f"総時間: {total/3600:.1f} 時間")
print(f"語彙サイズ: {len(labels)}")

# 3. 訓練データと検証データの分割 (SPLIT) を直接実行します
import random
random.seed(42)

lines = open(MANIFEST, encoding="utf-8").read().splitlines()
random.shuffle(lines)
n_val = max(200, int(len(lines) * 0.02))
val, train = lines[:n_val], lines[n_val:]

for name, rows in [("train", train), ("val", val)]:
    with open(f"/content/manifest_{name}.json", "w", encoding="utf-8") as f:
        f.write("\n".join(rows) + "\n")
    print(f"manifest_{name}.json: {len(rows)} 件")

ReazonSpeech データセットの読み込みを開始します...


Generating train split: 0 examples [00:00, ? examples/s]

  3%|▎         | 1947/62047 [00:06<03:10, 315.82it/s]

設定された制限時間 3.0 時間に達したため、抽出を終了します。

壊れて飛ばした: 0 件  []
条件で除外した: 17 件（無音/長すぎ/テキスト空）
発話数: 1931 → OOV除去後: 1025
総時間: 3.0 時間
語彙サイズ: 802
manifest_train.json: 825 件
manifest_val.json: 200 件
